# Save complete grounding labels — exact Output155
        Accelerator **None**, Internet ON. No Add Input. Enable secret `KAGGLE_API_TOKEN`
        belonging to the account that can read private source155; if named `codex`, change SECRET_NAME.
        Downloads full TRAIN label output, verifies reviewed summary/cohort/lexicon and copies
        unchanged into a NEW private Dataset. No training. Keep source155 until Dataset is ready.
        If creation/status fails, preserve payload; do not repeatedly create/overwrite.


In [ ]:
import hashlib,json,os,re,subprocess,sys,tempfile,urllib.request
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/155'
OWNER='ducanh2403'
SLUG='csa-grounding-audit30k-v155'
TITLE='CSA grounding audit30k v155 dataset'
SECRET_NAME='KAGGLE_API_TOKEN' # Change to codex if that is the secret's name.
DESTINATION=Path('/kaggle/working/csa_grounding_audit30k_v155_dataset')
SOURCE=Path(tempfile.mkdtemp(prefix='save-grounding155-'))


In [ ]:
COMMIT='85004b225198a0eeeb8d93f487f443d0573c7fe7'
HASHES={'package_grounding150.py': 'b29e1a4cdd8a8a3fe8317c2f9724b8d8983195aaeaba96210ae573aff1b18b4f', 'real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'grounding_audit_30k_review_text/summary.json': 'bfcbe0c861bec7770b004a866e0b78aadbbdfd1e8b418d2a5b47cf741bb4658f'}
for name,expected in HASHES.items():
    raw=urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{name}',timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Changed pinned source:'+name)
    p=SOURCE/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/155':raise ValueError('This notebook pins155; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/155', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v155', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v155; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
sys.path.insert(0,str(SOURCE))
from package_grounding150 import package,names_for
NAMES=names_for(30000)
DOWNLOAD=Path(tempfile.mkdtemp(prefix='audit155-download-'))
pattern=r'(^|/)(?:'+'|'.join(re.escape(n) for n in NAMES)+r')$'
run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),'--file-pattern',pattern])
package(DOWNLOAD,DESTINATION,study_images=30000,source_version=155)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private weak category-word/region labels from exact CSA audit output155. TRAIN30k only; original labels and provenance preserved. No model/checkpoint/features. Retains COCO upstream terms. Human label review pending.')
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; choose a fresh destination.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('Creation accepted; Dataset may still be processing. Do not repeatedly create.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',f'{OWNER}/{SLUG}'])
except RuntimeError:print('Status unconfirmed; check Dataset page before retrying.',flush=True)


Expected mount: `/kaggle/input/datasets/ducanh2403/csa-grounding-audit30k-v155`.
        Share this private Dataset and the CSA checkpoint/ROI inputs with the second account
        before running its GPU arm. Send actual mount if different.
